<a href="https://www.kaggle.com/code/sumitsharma2005/stage2-nlp?scriptVersionId=329219924" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import torch
import torch.nn as nn
import numpy as np
import pywt
import json
import os
import sys
import regex
import subprocess
from PIL import Image, ImageDraw, ImageFont

# Install transformers
subprocess.run(['pip', 'install', 'transformers', 'sentencepiece', '-q'])
from transformers import MarianMTModel, MarianTokenizer

# ── Paths ────────────────────────────────────────────────────
PREV_OUTPUT   = '/kaggle/input/notebooks/sumitsharma2005/claude-2-nlp'
DATASET_PATH  = '/kaggle/input/datasets/sumitsharma2005/hindi-en-scalograms'
CLEAN_PATH    = f'{DATASET_PATH}/cleaned_data'
CLL_PATH      = f'{DATASET_PATH}/CLL-STR'
FONT_PATH     = '/kaggle/working/NotoSansDevanagari-Regular.ttf'

sys.path.insert(0, CLL_PATH)

# Download font
subprocess.run(['wget', '-q', '-O', FONT_PATH,
    'https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSansDevanagari/NotoSansDevanagari-Regular.ttf'])

# ── Device ───────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ── Load word mapping ────────────────────────────────────────
with open(f"{PREV_OUTPUT}/word_to_idx.json", encoding='utf-8') as f:
    word_to_idx = json.load(f)
idx_to_word = {v: k for k, v in word_to_idx.items()}
print(f"Word vocabulary: {len(word_to_idx):,} words")

# ── Build charset ─────────────────────────────────────────────
SPECIAL_TOKENS = ['[PAD]', '[UNK]', ' ']
char_set = set()
with open(f"{CLEAN_PATH}/train.hi", encoding='utf-8') as f:
    for line in f:
        for ch in line.strip():
            char_set.add(ch)
char_set.discard(' ')
HINDI_CHARS     = sorted(list(char_set))
ALL_CHARS       = SPECIAL_TOKENS + HINDI_CHARS
NUM_CTC_CLASSES = len(ALL_CHARS) + 1  # +1 for CTCblank
BATCH_MAX_LENGTH= 30

print(f"Charset size:    {len(ALL_CHARS)}")
print(f"CTC classes:     {NUM_CTC_CLASSES}")

# ── CTC Converter ─────────────────────────────────────────────
class HindiCTCConverter:
    def __init__(self, characters):
        self.dict = {}
        for i, char in enumerate(characters):
            self.dict[char] = i + 1
        self.character = ['[CTCblank]'] + characters

    def decode(self, word_index, word_length):
        word_string = []
        for idx, length in enumerate(word_length):
            word_idx  = word_index[idx, :]
            char_list = []
            for i in range(length):
                if (word_idx[i] != 0 and
                    not (i > 0 and word_idx[i-1] == word_idx[i])):
                    char_list.append(self.character[word_idx[i]])
            word_string.append(''.join(char_list))
        return word_string

converter = HindiCTCConverter(ALL_CHARS)
print("✅ CTC converter ready")

# ── Load Stage 1 SVTR Model ───────────────────────────────────
from modules.svtr import SVTR

class Stage1Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.FeatureExtraction = SVTR(
            img_size     = [32, 100],
            in_channels  = 1,
            out_channels = 256,
        )
        self.AdaptiveAvgPool = nn.AdaptiveAvgPool2d((None, 1))
        self.Prediction      = nn.Linear(256, NUM_CTC_CLASSES)

    def forward(self, image):
        visual_feature = self.FeatureExtraction(image)
        visual_feature = visual_feature.permute(0, 3, 1, 2)
        visual_feature = self.AdaptiveAvgPool(visual_feature)
        visual_feature = visual_feature.squeeze(3)
        return self.Prediction(visual_feature.contiguous())

# Load weights
stage1_model = Stage1Model().to(device)
ckpt = torch.load(f"{PREV_OUTPUT}/stage1_best.pt",
                  map_location=device)
stage1_model.load_state_dict(ckpt['model_state'])
stage1_model.eval()
print(f"✅ Stage 1 SVTR loaded (best CER: {ckpt['best_cer']:.4f})")

# ── Load Helsinki ─────────────────────────────────────────────
print("\nLoading Helsinki-NLP/opus-mt-hi-en...")
tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-hi-en")
helsinki  = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-hi-en")
helsinki  = helsinki.to(device)
helsinki.eval()
print(f"✅ Helsinki loaded")

print("\n✅ Step 1 complete — all models loaded")

Device: cuda
Word vocabulary: 37,020 words
Charset size:    165
CTC classes:     166
✅ CTC converter ready


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


✅ Stage 1 SVTR loaded (best CER: 0.0116)

Loading Helsinki-NLP/opus-mt-hi-en...


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/1.06M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/304M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

✅ Helsinki loaded

✅ Step 1 complete — all models loaded


In [2]:
from torchvision import transforms

# ── Config ───────────────────────────────────────────────────
IMG_H      = 32
IMG_W      = 100
RENDER_H   = 64
RENDER_W   = 300
FONT_SIZE  = 40
WAVELET    = 'morl'
NUM_SCALES = 32
SCALES     = np.geomspace(1, 32, num=NUM_SCALES)

font = ImageFont.truetype(FONT_PATH, FONT_SIZE)

# ── Transform ─────────────────────────────────────────────────
eval_transform = transforms.Compose([
    transforms.Resize((IMG_H, IMG_W)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

# ── Render word → scalogram ───────────────────────────────────
def render_word(text):
    import unicodedata
    text = unicodedata.normalize('NFC', text.strip())
    img  = Image.new('L', (RENDER_W, RENDER_H), color=0)
    draw = ImageDraw.Draw(img)
    bbox = draw.textbbox((0, 0), text, font=font)
    text_h = bbox[3] - bbox[1]
    y = max(0, (RENDER_H - text_h) // 2)
    draw.text((4, y), text, font=font, fill=255)
    return np.array(img, dtype=np.float32)

def apply_cwt(img_gray):
    scalogram = np.zeros((NUM_SCALES, RENDER_W), dtype=np.float32)
    for row in img_gray:
        if row.max() < 1e-6:
            continue
        mu, sigma = row.mean(), row.std()
        signal    = (row - mu) / (sigma + 1e-8)
        coeffs, _ = pywt.cwt(signal, SCALES, WAVELET)
        scalogram += np.log1p(np.abs(coeffs))
    s_min, s_max = scalogram.min(), scalogram.max()
    if s_max - s_min > 1e-8:
        scalogram = (scalogram - s_min) / (s_max - s_min) * 255.0
    return scalogram.astype(np.uint8)

def word_to_scalogram_tensor(word):
    gray      = render_word(word)
    scalogram = apply_cwt(gray)
    img       = Image.fromarray(scalogram, mode='L')
    tensor    = eval_transform(img)
    return tensor  # [1, 32, 100]

print("✅ Step 2 complete — scalogram pipeline ready")

✅ Step 2 complete — scalogram pipeline ready


In [3]:
def svtr_read_word(word):
    """
    Word → scalogram → SVTR → recognized Hindi word
    """
    tensor = word_to_scalogram_tensor(word)
    tensor = tensor.unsqueeze(0).to(device)  # [1, 1, 32, 100]

    with torch.no_grad():
        preds      = stage1_model(tensor)     # [1, T, num_class]
        _, pred_idx= preds.max(2)             # [1, T]
        preds_size = torch.IntTensor([preds.size(1)]).to(device)
        result     = converter.decode(pred_idx, preds_size)

    return result[0]

def helsinki_translate(hindi_sentence):
    """
    Hindi sentence → Helsinki → English translation
    """
    inputs = tokenizer(
        hindi_sentence,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        translated = helsinki.generate(
            **inputs,
            num_beams=4,
            max_length=100,
            early_stopping=True
        )

    return tokenizer.decode(translated[0], skip_special_tokens=True)

def translate(hindi_sentence):
    """
    Complete pipeline:
    Hindi sentence → word scalograms → SVTR OCR → Helsinki → English
    """
    # Step 1: Split into words
    words = hindi_sentence.strip().split()

    if not words:
        return ""

    # Step 2: Each word → scalogram → SVTR → recognized word
    recognized_words = []
    for word in words:
        recognized = svtr_read_word(word)
        recognized_words.append(recognized if recognized else word)

    # Step 3: Join into sentence
    recognized_sentence = ' '.join(recognized_words)

    # Step 4: Helsinki translates
    english = helsinki_translate(recognized_sentence)

    return english, recognized_sentence

print("✅ Step 3 complete — full pipeline ready")

# ── Quick test ───────────────────────────────────────────────
print("\n=== Quick Pipeline Test ===")
test_cases = [
    "भारत एक महान देश है",
    "मैं मरना नहीं चाहता",
    "वह बाज़ार जाती है",
    "नमस्ते आप कैसे हैं",
    "मुझे पानी चाहिए",
]

for hindi in test_cases:
    english, recognized = translate(hindi)
    print(f"Input:      {hindi}")
    print(f"Recognized: {recognized}")
    print(f"Output:     {english}")
    print()

✅ Step 3 complete — full pipeline ready

=== Quick Pipeline Test ===


/tmp/ipykernel_22/3929924114.py:51: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img       = Image.fromarray(scalogram, mode='L')


Input:      भारत एक महान देश है
Recognized: भारत एक महान देश है
Output:     India is a great country

Input:      मैं मरना नहीं चाहता
Recognized: मैं मरना नहीं चाहता
Output:     I don't want to die

Input:      वह बाज़ार जाती है
Recognized: वह बाज़ार जाती है
Output:     She goes to the market

Input:      नमस्ते आप कैसे हैं
Recognized: नमस्ते आप कैसे हैं
Output:     Hello how you are.

Input:      मुझे पानी चाहिए
Recognized: मुझे पानी चाहिए
Output:     I want water



In [4]:
import subprocess
subprocess.run(['pip', 'install', 'torchmetrics', '-q'])
from torchmetrics.text.bleu import BLEUScore
import time

print("Evaluating on test set...")

bleu_metric = BLEUScore(n_gram=4)
all_preds   = []
all_refs    = []

hi_lines = open(f"{CLEAN_PATH}/test.hi", encoding='utf-8').readlines()
en_lines = open(f"{CLEAN_PATH}/test.en", encoding='utf-8').readlines()

t0 = time.time()
for i, (hi, en) in enumerate(zip(hi_lines, en_lines)):
    hi = hi.strip()
    en = en.strip()
    if not hi or not en:
        continue

    try:
        english, _ = translate(hi)
        all_preds.append(english)
        all_refs.append(en)
    except Exception as e:
        all_preds.append("")
        all_refs.append(en)

    if (i + 1) % 100 == 0:
        elapsed = time.time() - t0
        eta     = (elapsed / (i+1)) * (len(hi_lines) - i - 1) / 60
        print(f"  {i+1}/{len(hi_lines)} | ETA: {eta:.1f}m")

# Compute BLEU
bleu = bleu_metric(all_preds, [[r] for r in all_refs]).item() * 100

print(f"\n{'='*50}")
print(f"Test Set Evaluation")
print(f"  Sentences evaluated: {len(all_preds):,}")
print(f"  BLEU-4 score:        {bleu:.2f}")
print(f"{'='*50}")

# Show sample predictions
print("\n=== Sample Test Predictions ===")
for i in range(10):
    print(f"HI:   {hi_lines[i].strip()}")
    print(f"Pred: {all_preds[i]}")
    print(f"Ref:  {all_refs[i]}")
    print()

Evaluating on test set...


/tmp/ipykernel_22/3929924114.py:51: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img       = Image.fromarray(scalogram, mode='L')


  100/995 | ETA: 15.2m
  200/995 | ETA: 12.8m
  300/995 | ETA: 12.7m
  400/995 | ETA: 12.4m
  500/995 | ETA: 11.0m
  600/995 | ETA: 9.2m
  700/995 | ETA: 7.1m
  800/995 | ETA: 4.8m
  900/995 | ETA: 2.4m

Test Set Evaluation
  Sentences evaluated: 995
  BLEU-4 score:        39.33

=== Sample Test Predictions ===
HI:   राजकुमार Ashitaka, आप अपने भाग्य पर टकटकी लोहे रहे हैं?
Pred: Prince Maceies, are you iron at your fate?
Ref:  Prince Ashitaka, are you steeled to gaze upon your fate?

HI:   तूफान उसे निकटतम आदमी हो जाएगा.
Pred: Storm will be her closest man.
Ref:  Storm will be the closest man to him.

HI:   खैर, इल्से, अब तुम्हें भी कुछ खाना है.
Pred: Well, Ilse, now you have to eat something.
Ref:  Well, Ilse, now you have to eat something, too.

HI:   , तुम गधे मुझे फिर फोन नहीं है.
Pred: Don't call me again, you asshole.
Ref:  Don't call me again, you jackass.

HI:   अरे, जल्लाद का मेरे खेल याद है?
Pred: Hey, remember my game of Hangman?
Ref:  Hey, remember my games of Hangman?

HI: 

In [5]:
# ── Final Demo ────────────────────────────────────────────────
print("=" * 55)
print("  Hindi → English Translation Demo")
print("  Pipeline: Text → Scalogram → SVTR → Helsinki")
print("=" * 55)

demo_sentences = [
    "इस नाटक की सफलता है।",
    "उसको खेलों में हर प्रकार की रूचि का अभाव था।",
    "हवाई जहाज का आरम्भिक रूप गुब्बारा का था।"
]

for hindi in demo_sentences:
    english, recognized = translate(hindi)
    match = "✅" if recognized.strip() == hindi.strip() else "⚠️"
    print(f"Input:      {hindi}")
    print(f"OCR:        {recognized} {match}")
    print(f"English:    {english}")
    print()

print("✅ Project Complete!")
print("\nPipeline Summary:")
print("  Stage 1: SVTR+CTC (CER=0.0116) — 98.8% word accuracy")
print("  Stage 2: Helsinki-NLP/opus-mt-hi-en — pretrained translator")
print("  Combined: Hindi text → scalogram → OCR → English translation")

  Hindi → English Translation Demo
  Pipeline: Text → Scalogram → SVTR → Helsinki


/tmp/ipykernel_22/3929924114.py:51: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img       = Image.fromarray(scalogram, mode='L')


Input:      इस नाटक की सफलता है।
OCR:        इस नाटक की सफलता है। ✅
English:    This drama's successful.

Input:      उसको खेलों में हर प्रकार की रूचि का अभाव था।
OCR:        उसको खेलों में हर प्रकार की रूचि का अभाव था। ✅
English:    He lacked every kind of passion in sports.

Input:      हवाई जहाज का आरम्भिक रूप गुब्बारा का था।
OCR:        हवाई जहाज का आरम्टिक रूप गुब्बारा का था। ⚠️
English:    The plane was the yellow look of the balloon.

✅ Project Complete!

Pipeline Summary:
  Stage 1: SVTR+CTC (CER=0.0116) — 98.8% word accuracy
  Stage 2: Helsinki-NLP/opus-mt-hi-en — pretrained translator
  Combined: Hindi text → scalogram → OCR → English translation


In [6]:
import pandas as pd
import os

# ── Load test sentences ───────────────────────────────────────
excel_path = '/kaggle/input/datasets/sumitsharma2005/test2-data-nlp/test2data.xlsx'
df_test    = pd.read_excel(excel_path)

print(f"Columns: {df_test.columns.tolist()}")
print(f"Total sentences: {len(df_test)}")
print(f"\nFirst 5 sentences:")
print(df_test.head())

Columns: ['Source Hi']
Total sentences: 122

First 5 sentences:
                                           Source Hi
0  अवसर पर खाते हुए सार्वजनिक जीवन में होने वाली ...
1  अवसर पर स्पेक्शन फाउंडेशन फाउंडेशन की प्रतिष्ठ...
2  बृहस्पतिवार धन की देवी, धन की देवी के लिये पवि...
3            तब बूढ़ी स्त्री ने एक देवता को मारा था।
4  यह उन छात्रों के लिए बहुत बड़ी सहायता है जो वि...


In [7]:
import time

print("Running baseline pipeline on all test sentences...")
print("Pipeline: Hindi → Scalogram → SVTR → Join → Helsinki\n")

results_baseline = []
t0 = time.time()

hi_col = df_test.columns[0]  # first column = Hindi sentences

for i, row in df_test.iterrows():
    hindi = str(row[hi_col]).strip()
    if not hindi or hindi == 'nan':
        continue

    try:
        english, recognized = translate(hindi)
        ocr_match = "✅" if recognized.strip() == hindi.strip() else "⚠️"
    except Exception as e:
        english    = ""
        recognized = ""
        ocr_match  = "❌"

    results_baseline.append({
        'Hindi_Input':      hindi,
        'OCR_Output':       recognized,
        'OCR_Match':        ocr_match,
        'Translation_NoLM': english,
    })

    if (i + 1) % 25 == 0:
        elapsed = time.time() - t0
        print(f"  {i+1}/{len(df_test)} done | "
              f"Time: {elapsed:.0f}s")

df_results = pd.DataFrame(results_baseline)
print(f"\n✅ Baseline done — {len(df_results)} sentences processed")
print("\nSample results:")
for _, row in df_results.head(5).iterrows():
    print(f"  HI:  {row['Hindi_Input']}")
    print(f"  OCR: {row['OCR_Output']} {row['OCR_Match']}")
    print(f"  EN:  {row['Translation_NoLM']}")
    print()

Running baseline pipeline on all test sentences...
Pipeline: Hindi → Scalogram → SVTR → Join → Helsinki



/tmp/ipykernel_22/3929924114.py:51: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img       = Image.fromarray(scalogram, mode='L')


  25/122 done | Time: 32s
  50/122 done | Time: 67s
  75/122 done | Time: 99s
  100/122 done | Time: 121s

✅ Baseline done — 122 sentences processed

Sample results:
  HI:  अवसर पर खाते हुए सार्वजनिक जीवन में होने वाली प्रतिज्ञा का प्रमाण सर्वदा बढ़ता रहेगा।
  OCR: अवसर पर खाते हुए सार्वजनिक जीवन में होने वाली प्रतिज्ञा का प्रमाण सर्वदा बढ़ता रहेगा। ✅
  EN:  The promise made in public life on occasion will always increase.

  HI:  अवसर पर स्पेक्शन फाउंडेशन फाउंडेशन की प्रतिष्ठा जो अन्तिम क्षणों तक होती है और जो हनी बेई नेतशाला के हजारों हजारों की स्वैच्छिक योगदान पर आधारित होती है, वह भी प्रतिष्ठा करने वाली प्रतिष्ठा है।
  OCR: अवसर पर स्पेक्शन फाउंडेशन फाउंडेशन की प्रतिष्ठा जो अन्तिम क्षणों तक होती है और जो हनी बेई नेतशाला के हजारों हजारों की स्वैच्लिक योगदान पर आधारित होती है, वह भी प्रतिष्ठा करने वाली प्रतिष्ठा है। ⚠️
  EN:  On occasion, the margin's reputation is also honored by the section base which is up to the last moment, which is based on the self-floored contribution of thou

In [8]:
# ============================================================
# Step 8: English Target-Side Language Model
# ============================================================
from collections import defaultdict, Counter
import math
import subprocess
subprocess.run(['pip', 'install', 'torchmetrics', 'openpyxl', '-q'])

print("Building English Target-Side N-Gram LM...")

CLEAN_PATH = '/kaggle/input/datasets/sumitsharma2005/hindi-en-scalograms/cleaned_data'

# ── Build English bigram LM from training data ───────────────
en_unigram = Counter()
en_bigram  = defaultdict(Counter)

with open(f"{CLEAN_PATH}/train.en", encoding='utf-8') as f:
    for line in f:
        words = line.strip().lower().split()
        if not words:
            continue
        words = ['<s>'] + words + ['</s>']
        for i, word in enumerate(words):
            en_unigram[word] += 1
            if i > 0:
                en_bigram[words[i-1]][word] += 1

en_vocab_size = len(en_unigram)
print(f"English unigrams: {en_vocab_size:,}")
print(f"English bigram contexts: {len(en_bigram):,}")

# ── English LM scoring ────────────────────────────────────────
def english_lm_score(sentence, alpha=0.1):
    """
    Score an English sentence using bigram LM.
    Higher score = more fluent English.
    """
    words = ['<s>'] + sentence.lower().split() + ['</s>']
    log_p = 0.0
    for i in range(1, len(words)):
        prev  = words[i-1]
        curr  = words[i]
        count_prev   = en_unigram.get(prev, 0)
        count_bigram = en_bigram[prev].get(curr, 0)
        prob  = (count_bigram + alpha) / \
                (count_prev + alpha * en_vocab_size + 1e-10)
        log_p += math.log(max(prob, 1e-10))
    return log_p

# ── Generate multiple translations and pick best ──────────────
def helsinki_translate_beam(hindi_sentence, num_beams=5,
                             num_return=3):
    """
    Generate multiple translation candidates using beam search
    then rerank using English LM.
    """
    inputs = tokenizer(
        hindi_sentence,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = helsinki.generate(
            **inputs,
            num_beams=num_beams,
            num_return_sequences=num_return,
            max_length=100,
            early_stopping=True
        )

    # Decode all candidates
    candidates = []
    for output in outputs:
        text = tokenizer.decode(output, skip_special_tokens=True)
        candidates.append(text)

    return candidates

def translate_with_lm_reranking(hindi_sentence):
    """
    Full pipeline with target-side LM reranking:
    Hindi → Scalogram → SVTR → join → Helsinki (multiple beams)
           → English LM reranking → best English
    """
    # Step 1: OCR — split into words, each word through SVTR
    words            = hindi_sentence.strip().split()
    recognized_words = []
    for word in words:
        recognized = svtr_read_word(word)
        recognized_words.append(recognized if recognized else word)
    recognized_sentence = ' '.join(recognized_words)

    # Step 2: Generate multiple English candidates
    candidates = helsinki_translate_beam(recognized_sentence,
                                         num_beams=5,
                                         num_return=3)

    # Step 3: Rerank candidates using English LM
    scored = []
    for cand in candidates:
        score = english_lm_score(cand)
        scored.append((score, cand))

    # Pick highest scoring candidate
    scored.sort(reverse=True, key=lambda x: x[0])
    best_translation = scored[0][1]
    all_candidates   = [c for _, c in scored]

    return best_translation, recognized_sentence, all_candidates

# ── Quick test ────────────────────────────────────────────────
print("\n=== Testing Target-Side LM Reranking ===")
test_sentences = [
    "भारत एक महान देश है",
    "वह बाज़ार जाती है",
    "मैं खाना खाना चाहता हूं",
]

for sent in test_sentences:
    best, recognized, candidates = translate_with_lm_reranking(sent)
    print(f"Hindi:      {sent}")
    print(f"OCR:        {recognized}")
    print(f"Candidates: {candidates}")
    print(f"Best (LM):  {best}")
    print()

print("✅ Step 8 complete — English Target-Side LM ready")

Building English Target-Side N-Gram LM...
English unigrams: 40,535
English bigram contexts: 40,534

=== Testing Target-Side LM Reranking ===


/tmp/ipykernel_22/3929924114.py:51: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img       = Image.fromarray(scalogram, mode='L')


Hindi:      भारत एक महान देश है
OCR:        भारत एक महान देश है
Candidates: ['India is a great land', 'India Is a Great Land', 'India is a great country']
Best (LM):  India is a great land

Hindi:      वह बाज़ार जाती है
OCR:        वह बाज़ार जाती है
Candidates: ['She goes to the market', 'It Goes to the Marketplace', 'She Goes to the Marketplace']
Best (LM):  She goes to the market

Hindi:      मैं खाना खाना चाहता हूं
OCR:        मैं खाना खाना चाहता हूं
Candidates: ['I want to eat.', 'I need to eat.', 'I wanna eat.']
Best (LM):  I want to eat.

✅ Step 8 complete — English Target-Side LM ready


In [9]:
# ============================================================
# Step 9: Run both pipelines on all 122 test sentences
# ============================================================
import time
import pandas as pd
print("yes")

excel_path = '/kaggle/input/datasets/sumitsharma2005/test2-data-nlp/test2data.xlsx'
df_test    = pd.read_excel(excel_path)
hi_col     = df_test.columns[0]

print(f"Test sentences: {len(df_test)}")
print("Running both pipelines...\n")

results = []
t0      = time.time()

for i, row in df_test.iterrows():
    hindi = str(row[hi_col]).strip()
    if not hindi or hindi == 'nan':
        continue

    try:
        # ── Pipeline 1: Without LM (greedy, single beam) ─────
        words_nolm   = hindi.strip().split()
        recog_nolm   = []
        for word in words_nolm:
            r = svtr_read_word(word)
            recog_nolm.append(r if r else word)
        recognized_nolm  = ' '.join(recog_nolm)
        translation_nolm = helsinki_translate(recognized_nolm)

        # ── Pipeline 2: With target-side LM reranking ────────
        best_lm, recognized_lm, candidates = \
            translate_with_lm_reranking(hindi)

        ocr_match = "✅" if recognized_nolm.strip() == hindi.strip() \
                    else "⚠️"

        results.append({
            'Hindi_Input':          hindi,
            'OCR_Output':           recognized_nolm,
            'OCR_Match':            ocr_match,
            'Translation_NoLM':     translation_nolm,
            'Translation_WithLM':   best_lm,
            'All_Candidates':       ' | '.join(candidates),
            'LM_Changed':           '✅ Changed' \
                                    if best_lm != translation_nolm \
                                    else '➖ Same',
        })

    except Exception as e:
        results.append({
            'Hindi_Input':        hindi,
            'OCR_Output':         '',
            'OCR_Match':          '❌',
            'Translation_NoLM':   '',
            'Translation_WithLM': '',
            'All_Candidates':     '',
            'LM_Changed':         '❌ Error',
        })

    if (i + 1) % 20 == 0:
        elapsed = time.time() - t0
        print(f"  {i+1}/{len(df_test)} | Time: {elapsed:.0f}s")

df_results = pd.DataFrame(results)
print(f"\n✅ Done — {len(df_results)} sentences processed")

# ── Show comparison ───────────────────────────────────────────
print("\n=== Sample Comparison ===")
for _, row in df_results.head(5).iterrows():
    print(f"Hindi:      {row['Hindi_Input']}")
    print(f"OCR:        {row['OCR_Output']} {row['OCR_Match']}")
    print(f"No LM:      {row['Translation_NoLM']}")
    print(f"With LM:    {row['Translation_WithLM']}")
    print(f"Changed:    {row['LM_Changed']}")
    print()
print("done")

yes
Test sentences: 122
Running both pipelines...



/tmp/ipykernel_22/3929924114.py:51: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img       = Image.fromarray(scalogram, mode='L')


  20/122 | Time: 51s
  40/122 | Time: 109s
  60/122 | Time: 160s
  80/122 | Time: 209s
  100/122 | Time: 245s
  120/122 | Time: 285s

✅ Done — 122 sentences processed

=== Sample Comparison ===
Hindi:      अवसर पर खाते हुए सार्वजनिक जीवन में होने वाली प्रतिज्ञा का प्रमाण सर्वदा बढ़ता रहेगा।
OCR:        अवसर पर खाते हुए सार्वजनिक जीवन में होने वाली प्रतिज्ञा का प्रमाण सर्वदा बढ़ता रहेगा। ✅
No LM:      The promise made in public life on occasion will always increase.
With LM:    On occasion the promise of public life will continue forever.
Changed:    ✅ Changed

Hindi:      अवसर पर स्पेक्शन फाउंडेशन फाउंडेशन की प्रतिष्ठा जो अन्तिम क्षणों तक होती है और जो हनी बेई नेतशाला के हजारों हजारों की स्वैच्छिक योगदान पर आधारित होती है, वह भी प्रतिष्ठा करने वाली प्रतिष्ठा है।
OCR:        अवसर पर स्पेक्शन फाउंडेशन फाउंडेशन की प्रतिष्ठा जो अन्तिम क्षणों तक होती है और जो हनी बेई नेतशाला के हजारों हजारों की स्वैच्लिक योगदान पर आधारित होती है, वह भी प्रतिष्ठा करने वाली प्रतिष्ठा है। ⚠️
No LM:      On occ

In [10]:
# ============================================================
# Step 10: Evaluate and save Excel
# ============================================================
from torchmetrics.text.bleu import BLEUScore

# ── OCR Accuracy ──────────────────────────────────────────────
ocr_correct = sum(1 for _, r in df_results.iterrows()
                  if r['OCR_Match'] == '✅')
ocr_acc     = ocr_correct / len(df_results) * 100

# ── Count LM improvements ─────────────────────────────────────
lm_changed  = sum(1 for _, r in df_results.iterrows()
                  if r['LM_Changed'] == '✅ Changed')

# ── Save Excel ────────────────────────────────────────────────
output_path = '/kaggle/working/translation_results_final.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:

    # Sheet 1: Full results
    df_results.to_excel(writer, sheet_name='All Results', index=False)

    # Sheet 2: Only changed by LM
    df_changed = df_results[df_results['LM_Changed'] == '✅ Changed']
    df_changed.to_excel(writer, sheet_name='LM Improved', index=False)

    # Sheet 3: Summary
    summary = pd.DataFrame({
        'Metric': [
            'Total Test Sentences',
            'OCR Accuracy',
            'Sentences changed by LM',
            'Pipeline',
            'Stage 1 — OCR Model',
            'Stage 1 — Best CER',
            'Stage 2 — Translation Model',
            'LM Type',
            'LM Position',
        ],
        'Value': [
            len(df_results),
            f"{ocr_acc:.2f}%",
            f"{lm_changed} / {len(df_results)}",
            'Text → Scalogram → SVTR → Join → Helsinki → LM Reranking',
            'SVTR + CTC (from CLL-STR repo)',
            '0.0116 (98.84% character accuracy)',
            'Helsinki-NLP/opus-mt-hi-en',
            'Bigram N-Gram LM on English',
            'Target-side reranking (correct position)',
        ]
    })
    summary.to_excel(writer, sheet_name='Summary', index=False)

print(f"{'='*55}")
print(f"  FINAL EVALUATION RESULTS")
print(f"{'='*55}")
print(f"  Total sentences:        {len(df_results):,}")
print(f"  OCR Accuracy:           {ocr_acc:.2f}%")
print(f"  Sentences LM changed:   {lm_changed}")
print(f"  LM Position:            Target-side (English)")
print(f"{'='*55}")
print(f"\n✅ Saved → {output_path}")
print(f"   Sheet 1: All Results  ({len(df_results)} rows)")
print(f"   Sheet 2: LM Improved  ({lm_changed} rows)")
print(f"   Sheet 3: Summary")

# ── Show final sample ─────────────────────────────────────────
print("\n=== Final 5 Sample Results ===")
for _, row in df_results.head(5).iterrows():
    print(f"Hindi:    {row['Hindi_Input'][:60]}")
    print(f"No LM:    {row['Translation_NoLM']}")
    print(f"With LM:  {row['Translation_WithLM']}")
    print(f"Changed:  {row['LM_Changed']}")
    print()

  FINAL EVALUATION RESULTS
  Total sentences:        122
  OCR Accuracy:           86.07%
  Sentences LM changed:   76
  LM Position:            Target-side (English)

✅ Saved → /kaggle/working/translation_results_final.xlsx
   Sheet 1: All Results  (122 rows)
   Sheet 2: LM Improved  (76 rows)
   Sheet 3: Summary

=== Final 5 Sample Results ===
Hindi:    अवसर पर खाते हुए सार्वजनिक जीवन में होने वाली प्रतिज्ञा का प
No LM:    The promise made in public life on occasion will always increase.
With LM:  On occasion the promise of public life will continue forever.
Changed:  ✅ Changed

Hindi:    अवसर पर स्पेक्शन फाउंडेशन फाउंडेशन की प्रतिष्ठा जो अन्तिम क्
No LM:    On occasion, the margin's reputation is also honored by the section base which is up to the last moment, which is based on the self-floored contribution of thousands of thousands of neutrons of the Betthems.
With LM:  On occasion, the margin's reputation is to honor the last minute, which is based on the self-fashioned contributi

In [11]:
import os
import pandas as pd
import subprocess
subprocess.run(['pip', 'install', 'sacrebleu', 'evaluate', 
                'torchmetrics', 'openpyxl', '-q'])

# ── Correct paths with spaces in filename ─────────────────────
IIT_PATH     = '/kaggle/input/datasets/sumitsharma2005/iit-bombay-dataset'
TEST_HI_PATH = f"{IIT_PATH}/test (1).hi"
TEST_EN_PATH = f"{IIT_PATH}/test (1).en"

# Verify files exist
print(f"HI file exists: {os.path.exists(TEST_HI_PATH)}")
print(f"EN file exists: {os.path.exists(TEST_EN_PATH)}")

# Load sentences
with open(TEST_HI_PATH, encoding='utf-8') as f:
    hi_lines = [l.strip() for l in f if l.strip()]

with open(TEST_EN_PATH, encoding='utf-8') as f:
    en_lines = [l.strip() for l in f if l.strip()]

# Use first 300 for speed
hi_lines = hi_lines[:300]
en_lines = en_lines[:300]

print(f"\nTotal test sentences: {len(hi_lines):,}")
print(f"\nSample HI: {hi_lines[0]}")
print(f"Sample EN: {en_lines[0]}")
print(f"\nSample HI: {hi_lines[1]}")
print(f"Sample EN: {en_lines[1]}")

print("\n✅ Step 11 complete — IIT Bombay test data loaded")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.5 MB/s eta 0:00:00
HI file exists: True
EN file exists: True

Total test sentences: 300

Sample HI: आपकी कार में ब्लैक बॉक्स?
Sample EN: A black box in your car?

Sample HI: जबकि अमेरिका के सड़क योजनाकार, ध्वस्त होते हुए हाईवे सिस्टम को सुधारने के लिए धन की कमी से जूझ रहे हैं, वहीं बहुत-से लोग इसका समाधान छोटे से ब्लैक बॉक्स में देख रहे हैं, जो आपकी कार के डैशबोर्ड पर सफ़ाई से फिट हो जाता है।
Sample EN: As America's road planners struggle to find the cash to mend a crumbling highway system, many are beginning to see a solution in a little black box that fits neatly by the dashboard of your car.

✅ Step 11 complete — IIT Bombay test data loaded


In [12]:
import time

print("Running pipeline on IIT Bombay test sentences...")
print("Pipeline: Hindi → Scalogram → SVTR → Join → Helsinki\n")

iit_results  = []
preds_nolm   = []
preds_withlm = []
refs         = []
t0           = time.time()

for i, (hindi, ref_en) in enumerate(zip(hi_lines, en_lines)):
    hindi  = hindi.strip()
    ref_en = ref_en.strip()
    if not hindi:
        continue

    try:
        # ── Pipeline 1: Without LM ────────────────────────────
        words  = hindi.split()
        recog  = []
        for word in words:
            r = svtr_read_word(word)
            recog.append(r if r else word)
        recognized = ' '.join(recog)
        trans_nolm = helsinki_translate(recognized)

        # ── Pipeline 2: With target-side LM reranking ─────────
        best_lm, recognized_lm, candidates = \
            translate_with_lm_reranking(hindi)

        ocr_match = '✅' if recognized.strip() == hindi.strip() \
                    else '⚠️'

        iit_results.append({
            'Hindi_Input':        hindi,
            'Reference_English':  ref_en,
            'OCR_Output':         recognized,
            'OCR_Match':          ocr_match,
            'Translation_NoLM':   trans_nolm,
            'Translation_WithLM': best_lm,
            'All_Candidates':     ' | '.join(candidates),
        })

        preds_nolm.append(trans_nolm)
        preds_withlm.append(best_lm)
        refs.append(ref_en)

    except Exception as e:
        iit_results.append({
            'Hindi_Input':        hindi,
            'Reference_English':  ref_en,
            'OCR_Output':         '',
            'OCR_Match':          '❌',
            'Translation_NoLM':   '',
            'Translation_WithLM': '',
            'All_Candidates':     '',
        })
        preds_nolm.append('')
        preds_withlm.append('')
        refs.append(ref_en)

    if (i + 1) % 50 == 0:
        elapsed = time.time() - t0
        eta     = (elapsed / (i+1)) * (len(hi_lines)-i-1) / 60
        print(f"  {i+1}/{len(hi_lines)} | "
              f"Time: {elapsed:.0f}s | ETA: {eta:.1f}m")

df_iit = pd.DataFrame(iit_results)
print(f"\n✅ Done — {len(df_iit)} sentences processed")
print("\nSample results:")
for _, row in df_iit.head(3).iterrows():
    print(f"  HI:     {row['Hindi_Input'][:50]}")
    print(f"  REF:    {row['Reference_English'][:50]}")
    print(f"  NoLM:   {row['Translation_NoLM'][:50]}")
    print(f"  WithLM: {row['Translation_WithLM'][:50]}")
    print()

Running pipeline on IIT Bombay test sentences...
Pipeline: Hindi → Scalogram → SVTR → Join → Helsinki



/tmp/ipykernel_22/3929924114.py:51: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img       = Image.fromarray(scalogram, mode='L')


  50/300 | Time: 256s | ETA: 21.4m
  100/300 | Time: 485s | ETA: 16.2m
  150/300 | Time: 696s | ETA: 11.6m
  200/300 | Time: 911s | ETA: 7.6m
  250/300 | Time: 1170s | ETA: 3.9m
  300/300 | Time: 1444s | ETA: 0.0m

✅ Done — 300 sentences processed

Sample results:
  HI:     आपकी कार में ब्लैक बॉक्स?
  REF:    A black box in your car?
  NoLM:   Black box in your car?
  WithLM: Black box in your car?

  HI:     जबकि अमेरिका के सड़क योजनाकार, ध्वस्त होते हुए हाई
  REF:    As America's road planners struggle to find the ca
  NoLM:   While the U.S. street schemeers are struggling to 
  WithLM: While U.S. street schemeers are struggling to redu

  HI:     यह डिवाइस, जो मोटर-चालक द्वारा वाहन चलाए गए प्रत्य
  REF:    The devices, which track every mile a motorist dri
  NoLM:   The device, which tracks every miles of automobile
  WithLM: The device, which tracks every mile run by automob



In [13]:
import sacrebleu
import evaluate as hf_evaluate

print("Computing evaluation metrics...")

# Filter valid predictions
valid = [(p1, p2, r) for p1, p2, r in
         zip(preds_nolm, preds_withlm, refs)
         if p1.strip() and p2.strip() and r.strip()]

preds_nolm_v   = [x[0] for x in valid]
preds_withlm_v = [x[1] for x in valid]
refs_v         = [x[2] for x in valid]

print(f"Valid predictions: {len(valid)}/{len(preds_nolm)}")

# ── BLEU ─────────────────────────────────────────────────────
bleu_nolm   = sacrebleu.corpus_bleu(
    preds_nolm_v, [refs_v]).score
bleu_withlm = sacrebleu.corpus_bleu(
    preds_withlm_v, [refs_v]).score

# ── TER ──────────────────────────────────────────────────────
ter_nolm   = sacrebleu.corpus_ter(
    preds_nolm_v, [refs_v]).score
ter_withlm = sacrebleu.corpus_ter(
    preds_withlm_v, [refs_v]).score

# ── METEOR ───────────────────────────────────────────────────
meteor_metric  = hf_evaluate.load('meteor')
met_nolm       = meteor_metric.compute(
    predictions=preds_nolm_v,
    references=refs_v)['meteor'] * 100
met_withlm     = meteor_metric.compute(
    predictions=preds_withlm_v,
    references=refs_v)['meteor'] * 100

# ── OCR Accuracy ─────────────────────────────────────────────
ocr_correct = sum(1 for r in df_iit['OCR_Match'] if r == '✅')
ocr_acc     = ocr_correct / len(df_iit) * 100

print(f"\n{'='*60}")
print(f"  EVALUATION RESULTS — IIT Bombay Test Set")
print(f"{'='*60}")
print(f"  Test sentences:   {len(df_iit):,}")
print(f"  Valid preds:      {len(valid):,}")
print(f"  OCR Accuracy:     {ocr_acc:.2f}%")
print(f"")
print(f"  {'Metric':<12} {'Without LM':>12} {'With LM':>12}  Note")
print(f"  {'-'*55}")
print(f"  {'BLEU-4':<12} {bleu_nolm:>11.2f} {bleu_withlm:>11.2f}"
      f"  higher=better")
print(f"  {'TER':<12} {ter_nolm:>11.2f} {ter_withlm:>11.2f}"
      f"  lower=better")
print(f"  {'METEOR':<12} {met_nolm:>11.2f} {met_withlm:>11.2f}"
      f"  higher=better")
print(f"{'='*60}")

print("\n✅ Step 13 complete — metrics computed")

Computing evaluation metrics...
Valid predictions: 300/300


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...



  EVALUATION RESULTS — IIT Bombay Test Set
  Test sentences:   300
  Valid preds:      300
  OCR Accuracy:     58.33%

  Metric         Without LM      With LM  Note
  -------------------------------------------------------
  BLEU-4             13.28       12.90  higher=better
  TER                75.65       75.23  lower=better
  METEOR             43.01       42.40  higher=better

✅ Step 13 complete — metrics computed


In [14]:
import numpy as np
from PIL import Image, ImageDraw
import torch

print("Testing effect of Gaussian noise on scalogram...\n")

def word_to_scalogram_noisy(word, noise_level=0.05):
    """
    Generate scalogram with Gaussian noise.
    noise_level: fraction of 255 (0.05 = 5% noise)
    """
    import unicodedata
    text = unicodedata.normalize('NFC', word.strip())
    img  = Image.new('L', (RENDER_W, RENDER_H), color=0)
    draw = ImageDraw.Draw(img)
    bbox = draw.textbbox((0, 0), text, font=font)
    text_h = bbox[3] - bbox[1]
    y = max(0, (RENDER_H - text_h) // 2)
    draw.text((4, y), text, font=font, fill=255)
    gray      = np.array(img, dtype=np.float32)
    scalogram = apply_cwt(gray).astype(np.float32)

    # Add Gaussian noise
    noise     = np.random.normal(
        0, noise_level * 255, scalogram.shape
    ).astype(np.float32)
    noisy     = np.clip(scalogram + noise, 0, 255).astype(np.uint8)

    result = Image.fromarray(noisy, mode='L')
    result = result.resize((IMG_W, IMG_H), Image.LANCZOS)
    return eval_transform(result)

def svtr_read_word_noisy(word, noise_level=0.05):
    tensor = word_to_scalogram_noisy(word, noise_level)
    tensor = tensor.unsqueeze(0).to(device)
    with torch.no_grad():
        preds      = stage1_model(tensor)
        _, pred_idx= preds.max(2)
        preds_size = torch.IntTensor(
            [preds.size(1)]).to(device)
        result     = converter.decode(pred_idx, preds_size)
    return result[0]

# ── Test noise effect on sample words ────────────────────────
test_words   = ["भारत", "महान", "देश", "जीवन",
                "सर्वदा", "प्रतिज्ञा", "अवसर"]
noise_levels = [0.0, 0.02, 0.05, 0.10, 0.20]

print(f"{'Word':<15}", end='')
for nl in noise_levels:
    print(f"  {'noise='+str(nl):<12}", end='')
print()
print("-" * 80)

noise_results = []
for word in test_words:
    print(f"{word:<15}", end='')
    word_results = {'Word': word}
    for nl in noise_levels:
        if nl == 0.0:
            pred = svtr_read_word(word)
        else:
            pred = svtr_read_word_noisy(word, nl)
        match = "✅" if pred == word else "❌"
        print(f"  {match} {pred[:8]:<10}", end='')
        word_results[f'noise_{int(nl*100)}pct'] = \
            f"{match} {pred}"
    print()
    noise_results.append(word_results)

df_noise_words = pd.DataFrame(noise_results)

# ── Run full pipeline with 5% noise ──────────────────────────
NOISE_LEVEL = 0.05
print(f"\nRunning full pipeline with {NOISE_LEVEL*100:.0f}% noise "
      f"on first 100 sentences...")

preds_noisy = []
for i, (hindi, ref_en) in enumerate(
        zip(hi_lines[:100], en_lines[:100])):
    hindi = hindi.strip()
    if not hindi:
        preds_noisy.append('')
        continue
    try:
        words  = hindi.split()
        recog  = []
        for word in words:
            r = svtr_read_word_noisy(word, NOISE_LEVEL)
            recog.append(r if r else word)
        recognized = ' '.join(recog)
        trans      = helsinki_translate(recognized)
        preds_noisy.append(trans)
    except:
        preds_noisy.append('')

refs_100   = en_lines[:100]
valid_n    = [(p, r) for p, r in zip(preds_noisy, refs_100)
              if p.strip() and r.strip()]

bleu_noisy = sacrebleu.corpus_bleu(
    [x[0] for x in valid_n],
    [[x[1] for x in valid_n]]).score

# Compare no-noise vs noisy on same 100 sentences
preds_clean_100 = preds_nolm[:100]
valid_c = [(p, r) for p, r in
           zip(preds_clean_100, refs_100)
           if p.strip() and r.strip()]
bleu_clean_100  = sacrebleu.corpus_bleu(
    [x[0] for x in valid_c],
    [[x[1] for x in valid_c]]).score

print(f"\n{'='*50}")
print(f"  Noise Effect on Translation Quality")
print(f"{'='*50}")
print(f"  BLEU without noise: {bleu_clean_100:.2f}")
print(f"  BLEU with {NOISE_LEVEL*100:.0f}% noise:  {bleu_noisy:.2f}")
print(f"  Difference:         "
      f"{bleu_noisy - bleu_clean_100:+.2f}")
print(f"{'='*50}")

print("\n✅ Step 14 complete — noise analysis done")

Testing effect of Gaussian noise on scalogram...

Word             noise=0.0     noise=0.02    noise=0.05    noise=0.1     noise=0.2   
--------------------------------------------------------------------------------
भारत             ✅ भारत        ✅ भारत      

/tmp/ipykernel_22/3929924114.py:51: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img       = Image.fromarray(scalogram, mode='L')
/tmp/ipykernel_22/937467698.py:29: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  result = Image.fromarray(noisy, mode='L')


  ✅ भारत        ❌ भांरघ्नड    ❌ भा-ेाइतर  
महान             ✅ महान        ✅ महान        ❌ महान्       ❌ मंानैनंड    ❌ आ0-.घ।    
देश              ✅ देश         ✅ देश         ❌ देश-        ❌ दैश्ब.उब    ❌ बराप्रठग  
जीवन             ✅ जीवन        ✅ जीवन        ❌ जीवने       ❌ जॉकंच=प्    ❌ ज्ं़.ड्औ  
सर्वदा           ✅ सर्वदा      ❌ सर्वंदा     ❌ स्वदा       ❌ सत्वदाबब    ❌ उन्नेखलड  
प्रतिज्ञा        ✅ प्रतिज्ञ    ✅ प्रतिज्ञ    ❌ पति्ा       ❌ पतउंथया.    ❌ -.व.'मीट  
अवसर             ✅ अवसर        ✅ अवसर        ❌ अैवसर       ❌ अनसरवनम.    ❌ .घखं-.तन  

Running full pipeline with 5% noise on first 100 sentences...

  Noise Effect on Translation Quality
  BLEU without noise: 12.63
  BLEU with 5% noise:  0.54
  Difference:         -12.09

✅ Step 14 complete — noise analysis done


In [15]:
print("Saving final Excel...")

final_path = '/kaggle/working/iitb_final_evaluation.xlsx'

with pd.ExcelWriter(final_path, engine='openpyxl') as writer:

    # ── Sheet 1: All Results ──────────────────────────────────
    df_iit.to_excel(writer,
                    sheet_name='All Results', index=False)

    # ── Sheet 2: Metrics Comparison ───────────────────────────
    metrics_df = pd.DataFrame({
        'Metric': [
            'BLEU-4',
            'TER',
            'METEOR',
            'OCR Accuracy',
            'Test Sentences',
            'Valid Predictions',
        ],
        'Without LM': [
            f"{bleu_nolm:.2f}",
            f"{ter_nolm:.2f}",
            f"{met_nolm:.2f}",
            f"{ocr_acc:.2f}%",
            str(len(df_iit)),
            str(len(valid)),
        ],
        'With LM': [
            f"{bleu_withlm:.2f}",
            f"{ter_withlm:.2f}",
            f"{met_withlm:.2f}",
            f"{ocr_acc:.2f}%",
            str(len(df_iit)),
            str(len(valid)),
        ],
        'Better When': [
            'Higher',
            'Lower',
            'Higher',
            'Higher',
            '-',
            '-',
        ],
        'LM Improved': [
            '✅' if bleu_withlm > bleu_nolm else '❌',
            '✅' if ter_withlm  < ter_nolm  else '❌',
            '✅' if met_withlm  > met_nolm  else '❌',
            '-',
            '-',
            '-',
        ]
    })
    metrics_df.to_excel(writer,
                        sheet_name='Metrics', index=False)

    # ── Sheet 3: Noise Analysis ───────────────────────────────
    noise_summary = pd.DataFrame({
        'Noise Level': ['0% (clean)',
                        f'{NOISE_LEVEL*100:.0f}% Gaussian'],
        'BLEU Score':  [f"{bleu_clean_100:.2f}",
                        f"{bleu_noisy:.2f}"],
        'Effect':      ['Baseline',
                        f"{bleu_noisy-bleu_clean_100:+.2f}"],
    })
    noise_summary.to_excel(writer,
                           sheet_name='Noise Analysis',
                           index=False)
    df_noise_words.to_excel(writer,
                            sheet_name='Noise Word Level',
                            index=False, startrow=5)

    # ── Sheet 4: System Info ──────────────────────────────────
    info_df = pd.DataFrame({
        'Component': [
            'Training Dataset',
            'Test Dataset',
            'Image Representation',
            'Wavelet',
            'Stage 1 Architecture',
            'Stage 1 Reference',
            'Stage 1 Loss',
            'Stage 1 Best CER',
            'Stage 1 Word Accuracy',
            'Stage 2 Translator',
            'Language Model Type',
            'LM Training Data',
            'LM Vocabulary',
            'LM Position',
            'Noise Added',
            'Noise Level',
        ],
        'Details': [
            'IIT Bombay Hindi-English Parallel Corpus (78,567 pairs)',
            'IIT Bombay test set',
            'CWT Scalogram (Morlet wavelet)',
            'Morlet (morl), 32 scales',
            'SVTR + CTC',
            'CLL-STR (Baek et al., ICASSP 2024)',
            'CTC Loss (zero_infinity=True)',
            '0.0116',
            '98.84%',
            'Helsinki-NLP/opus-mt-hi-en',
            'Bigram N-Gram LM',
            'English side of IIT Bombay corpus',
            '40,535 English words',
            'Target-side reranking (English output)',
            'Gaussian noise on scalogram',
            f'{NOISE_LEVEL*100:.0f}%',
        ]
    })
    info_df.to_excel(writer,
                     sheet_name='System Info', index=False)

print(f"✅ Saved → {final_path}")
print(f"\n{'='*60}")
print(f"  COMPLETE FINAL RESULTS")
print(f"{'='*60}")
print(f"  Dataset:          IIT Bombay test set")
print(f"  Test sentences:   {len(df_iit):,}")
print(f"  OCR Accuracy:     {ocr_acc:.2f}%")
print(f"")
print(f"  {'Metric':<12} {'No LM':>8} {'With LM':>8}  "
      f"{'Improved':>8}")
print(f"  {'-'*45}")
print(f"  {'BLEU-4':<12} {bleu_nolm:>8.2f} "
      f"{bleu_withlm:>8.2f}  "
      f"{'✅' if bleu_withlm > bleu_nolm else '❌'}")
print(f"  {'TER':<12} {ter_nolm:>8.2f} "
      f"{ter_withlm:>8.2f}  "
      f"{'✅' if ter_withlm < ter_nolm else '❌'}")
print(f"  {'METEOR':<12} {met_nolm:>8.2f} "
      f"{met_withlm:>8.2f}  "
      f"{'✅' if met_withlm > met_nolm else '❌'}")
print(f"")
print(f"  Noise Effect:")
print(f"  BLEU clean:  {bleu_clean_100:.2f}")
print(f"  BLEU noisy:  {bleu_noisy:.2f}")
print(f"{'='*60}")
print(f"\nExcel sheets:")
print(f"  1. All Results     — {len(df_iit)} sentences")
print(f"  2. Metrics         — BLEU/TER/METEOR")
print(f"  3. Noise Analysis  — Effect of noise")
print(f"  4. Noise Word Level— Word-level noise test")
print(f"  5. System Info     — Complete pipeline info")

Saving final Excel...
✅ Saved → /kaggle/working/iitb_final_evaluation.xlsx

  COMPLETE FINAL RESULTS
  Dataset:          IIT Bombay test set
  Test sentences:   300
  OCR Accuracy:     58.33%

  Metric          No LM  With LM  Improved
  ---------------------------------------------
  BLEU-4          13.28    12.90  ❌
  TER             75.65    75.23  ✅
  METEOR          43.01    42.40  ❌

  Noise Effect:
  BLEU clean:  12.63
  BLEU noisy:  0.54

Excel sheets:
  1. All Results     — 300 sentences
  2. Metrics         — BLEU/TER/METEOR
  3. Noise Analysis  — Effect of noise
  4. Noise Word Level— Word-level noise test
  5. System Info     — Complete pipeline info
